# Exploratory analysis - Student Risk Tracker

Inspects the bundled training data (`data/training/students_labeled.csv`): missing values, class balance, possible **target leakage**, and the relationships between attendance, assignment completion, previous scores and the risk label.

> Data is de-identified (no names or contact details). Findings describe *this dataset* only.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "training" / "students_labeled.csv")
CORE = ["Attendance_Percentage", "Assignment_Completion_Percentage", "Previous_Semester_GPA", "Previous_Assessment_Score"]
SUBJECTS = [c for c in df.columns if c.endswith("_Score") and c not in ("Previous_Assessment_Score", "Risk_Score")]
print(df.shape)
df.head()

## 1. Data quality

In [ ]:
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "none")
print("Duplicate Student_IDs:", df["Student_ID"].duplicated().sum())
df[CORE + SUBJECTS].describe().T[["min", "mean", "max"]].round(1)

## 2. Target definition and class balance
The dataset documents a `Risk_Score` (0-100). We bin it into **Low (<30), Moderate (30-49), High (>=50)** - High is identical to the dataset's own `At_Risk` flag, so no label is invented.

In [ ]:
df["Risk_Category"] = pd.cut(df["Risk_Score"], [-np.inf, 29.999, 49.999, np.inf], labels=["Low concern", "Moderate concern", "High concern"])
counts = df["Risk_Category"].value_counts().reindex(["Low concern", "Moderate concern", "High concern"])
print((counts / len(df) * 100).round(1))
counts.plot.bar(color=["#2e9e6b", "#f0a020", "#d64545"], title="Class balance"); plt.ylabel("Students"); plt.show()

## 3. Leakage check
`Risk_Score` and `At_Risk` *define* the label, so they must never be model inputs. Student_ID and demographic columns are also excluded.

In [ ]:
print("At_Risk == (Risk_Score >= 50) for every row:", bool((df["At_Risk"] == (df["Risk_Score"] >= 50)).all()))
print("Correlation of inputs with Risk_Score:")
df[CORE + SUBJECTS + ["Risk_Score"]].corr()["Risk_Score"].drop("Risk_Score").sort_values().round(3)

## 4. Attendance, assignments, GPA and assessments vs. risk

In [ ]:
colors = {"Low concern": "#2e9e6b", "Moderate concern": "#f0a020", "High concern": "#d64545"}
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), CORE):
    for cat, c in colors.items():
        ax.hist(df.loc[df["Risk_Category"] == cat, col], bins=25, alpha=.6, color=c, label=cat)
    ax.set_title(col.replace("_", " "))
axes[0, 0].legend(); plt.tight_layout(); plt.show()
df.groupby("Risk_Category", observed=True)[CORE].mean().round(2)

## 5. Subject-wise averages by risk category

In [ ]:
sub = df.groupby("Risk_Category", observed=True)[SUBJECTS].mean().round(1)
display(sub) if "display" in globals() else print(sub)
print("\nShare of students below 60 per subject (%):")
print((df[SUBJECTS] < 60).mean().mul(100).round(1).sort_values(ascending=False))

## 6. Correlation matrix

In [ ]:
corr = df[CORE + SUBJECTS + ["Risk_Score"]].corr()
fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(corr, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels([c.replace("_Score", "").replace("_Percentage", "") for c in corr.columns], rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels([c.replace("_Score", "").replace("_Percentage", "") for c in corr.columns])
plt.colorbar(im); plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

## Conclusions
* No missing values or duplicate IDs; High / Moderate / Low are roughly 35 % / 19 % / 46 % - moderately imbalanced, so recall and the confusion matrix are reported instead of accuracy alone.
* Attendance, previous GPA and assignment completion carry most of the signal; subject scores barely differ between categories.
* The label is a deterministic rule over a few indicators, so a flexible model will score near-perfectly on it. This reflects the data's construction, **not** real-world predictive power - see the README limitations.